# Austrian law RAG – 01: build the corpus, write questions, evaluate BM25

1. Downloads six Austrian federal laws (current version) from the RIS open-data API.
2. Lets you search and read §§ to write the evaluation questions.
3. Scores the BM25 baseline (and, optionally, a dense retriever and a hybrid).

Everything is saved to Google Drive (`MyDrive/austrian-law-rag`), so you can stop and continue later.
No GPU is needed for steps 1–3; the optional dense step is faster with *Runtime → T4 GPU*.

In [24]:
import os
if not os.path.exists('/content/austrian-law-rag'):
    !git clone -q https://github.com/zarekasif/austrian-law-rag.git /content/austrian-law-rag
else:
    !git -C /content/austrian-law-rag pull -q
%cd /content/austrian-law-rag
!pip install -q -e .

/content/austrian-law-rag
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for lawrag (pyproject.toml) ... done


In [25]:
from google.colab import drive
drive.mount('/content/drive')

DATA = '/content/drive/MyDrive/austrian-law-rag'
os.makedirs(DATA, exist_ok=True)
CORPUS = f'{DATA}/corpus.jsonl'
QUESTIONS = f'{DATA}/questions.jsonl'
if not os.path.exists(QUESTIONS):          # start from the example question in the repo
    !cp data/questions.jsonl "$QUESTIONS"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Build the corpus (first run about 10 minutes, then cached)

About 550 requests, one per second, as the RIS usage notes ask. Raw responses are cached in
Drive, so re-running is instant.

In [26]:
!python -m lawrag.corpus --out "$CORPUS" --cache "$DATA/raw"

UG       195 paragraphs
MRG       93 paragraphs
NAG       90 paragraphs
AZG       78 paragraphs
AuslBG    66 paragraphs
UrlG      23 paragraphs
total    545 paragraphs, version of 2026-11-02 -> /content/drive/MyDrive/austrian-law-rag/corpus.jsonl


In [27]:
import sys
import os

repo_path = '/content/austrian-law-rag'
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

In [28]:
import sys
import os

# Force locate and insert the parent directory of lawrag into sys.path
possible_paths = [
    '/content/austrian-law-rag',
    '/content/austrian-law-rag/src'
]
for path in possible_paths:
    if os.path.exists(path) and path not in sys.path:
        sys.path.insert(0, path)

import json, collections
from lawrag.retrieve import load_corpus, BM25Retriever
from lawrag.chunk import chunk_corpus

records = load_corpus(CORPUS)
by_id = {r['id']: r for r in records}
chunks = chunk_corpus(records)
bm25 = BM25Retriever(chunks)

print(len(records), 'paragraphs,', len(chunks), 'chunks')
print(collections.Counter(r['law'] for r in records))
longest = max(records, key=lambda r: len(r['text']))
print('longest:', longest['id'], len(longest['text']), 'characters')

545 paragraphs, 1435 chunks
Counter({'UG': 195, 'MRG': 93, 'NAG': 90, 'AZG': 78, 'AuslBG': 66, 'UrlG': 23})
longest: UG § 143 39575 characters


## 2. Write evaluation questions

Read `docs/writing-questions.md` first. Helpers:

- `browse('AZG')` lists every § of a law with its title,
- `show('AZG § 11')` prints a § in full,
- `search('Muss mein Chef mir eine Pause geben?')` shows BM25's top 10,
- `add('question', ['AZG § 11'], 'layperson')` saves a question to Drive.

Pick the gold § by **reading** (browse/show), not only from the search results.

In [ ]:
def browse(law):
    for r in records:
        if r['law'] == law:
            print(f"{r['id']:14s} {r['title']}")

def show(section_id):
    r = by_id[section_id]
    print(f"{r['id']} – {r['title']}\n{' › '.join(r['path'])}\n{r.get('url', '')}\n")
    print(r['text'])

def search(question, k=10):
    for rank, sid in enumerate(bm25(question)[:k], start=1):
        print(f"{rank:2d}. {sid:14s} {by_id[sid]['title']}")

def add(question, gold, qtype='layperson'):
    missing = [g for g in gold if g not in by_id]
    if missing:
        raise ValueError(f'not in corpus: {missing}')
    existing = [json.loads(l) for l in open(QUESTIONS, encoding='utf-8') if l.strip()]
    qid = f"q{len(existing) + 1:03d}"
    row = {'id': qid, 'question': question, 'gold': gold, 'law': gold[0].split()[0], 'type': qtype}
    with open(QUESTIONS, 'a', encoding='utf-8') as f:
        f.write(json.dumps(row, ensure_ascii=False) + '\n')
    print('saved', qid, '| BM25 rank of gold:',
          [bm25(question).index(g) + 1 if g in bm25(question) else None for g in gold])

browse('UrlG')

In [ ]:
show('MRG § 1')

In [ ]:
search('Gilt das Mietrecht auch für ein Zimmer im Studentenheim?')

In [ ]:
# Write your questions here, one add(...) per question, e.g.
# add('Muss mein Chef mir eine Pause geben, wenn ich 7 Stunden arbeite?', ['AZG § 11'])

## 3. Evaluate BM25

In [ ]:
!python -m lawrag.evaluate --corpus "$CORPUS" --questions "$QUESTIONS" --retrievers bm25 --out "$DATA/results_bm25.json"

## 4. Optional: dense retrieval and hybrid (multilingual E5)

Faster on a GPU runtime. Compare the three rows: where does BM25 fail and the dense model succeed,
and the other way round?

In [ ]:
!pip install -q -e ".[dense]"
!python -m lawrag.evaluate --corpus "$CORPUS" --questions "$QUESTIONS" --retrievers bm25 e5 hybrid --out "$DATA/results_all.json"